# Mission 1: The Overfitting Game

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/missions/mission_01_overfitting/notebook.ipynb)

**Learning objectives**
- Build intuition for in-sample vs. out-of-sample performance
- Experience the overfitting trap firsthand
- Submit a strategy and interpret your OOS grade report
- Understand Information Coefficient (IC) and its role in strategy evaluation

---

## Background

In quantitative finance, a **strategy** is a rule that ranks stocks and bets on the ranking being predictive of future returns.  The challenge: the market gives you historical data to *develop* your strategy (in-sample, IS), but profits only if the strategy works on *new* data it has never seen (out-of-sample, OOS).

Overfitting happens when you tune your strategy to historical noise.  It looks great in-sample and terrible out-of-sample.  Your mission today: experience this trap, then escape it.

---

## Part 0: Setup

In [ ]:
# Install convexpi-lab if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q convexpi-lab>=0.1.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from convexpi.lab import SyntheticMarket, Backtest, SimpleBacktest, Grader

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Ready.')

---
## Part 1: Explore the Synthetic Market

The ConvexPi Lab uses a **synthetic market** — a simulated panel of stock returns and features.  The market has 200 assets and 1,000 trading days (~4 years), and one of the features secretly predicts returns.  Your job is to find it.

The first 750 days are your **training window** (IS).  The final 250 days are the **hidden holdout** (OOS) — you never see those returns directly; the grader does.

In [ ]:
# Create a market with a default seed (the actual grader uses its own seed — this is for exploration)
market = SyntheticMarket(n_assets=200, n_days=1000, seed=42)

print(f"Assets  : {market.n_assets}")
print(f"Days    : {market.n_days}")
print(f"Train   : days 0..{market.train_end - 1}")
print(f"Holdout : days {market.train_end}..{market.n_days - 1}")
print(f"Features: {market.FEATURE_NAMES}")

In [ ]:
# Extract the training slice
train_features = market.features_array('train')   # shape: (train_days, n_assets, n_features)
train_returns  = market.returns('train')             # shape: (train_days-1, n_assets)

print(f"Features shape : {train_features.shape}")
print(f"Returns  shape : {train_returns.shape}")
print(f"Mean daily return: {train_returns.mean():.5f}")
print(f"Return std (cross-section): {train_returns.std(axis=1).mean():.5f}")

In [ ]:
# Visualise cumulative market return
cum_return = (1 + train_returns.mean(axis=1)).cumprod() - 1

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(cum_return * 100, color='steelblue', lw=1.5)
ax.axhline(0, color='grey', lw=0.5)
ax.set_title('Market cumulative return (equal-weighted, IS)')
ax.set_xlabel('Trading day')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()

---
## Part 2: Build a Baseline Strategy

A strategy is a Python class that ConvexPi's Lab expects to implement one method:

```python
class MyStrategy:
    def predict(self, features: np.ndarray) -> np.ndarray:
        """Return a 1-D score array of shape (n_assets,). Higher = more bullish."""
```

The backtester calls `predict()` every day, ranks the output into long/short positions, and records the realized P&L.

Let's start with a **random baseline** to understand what noise looks like, then improve it.

In [ ]:
class RandomStrategy:
    """Assign random scores — pure noise, no alpha."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)

    def predict(self, features: np.ndarray) -> np.ndarray:
        return self.rng.standard_normal(features.shape[0])


bt_random = SimpleBacktest(market, RandomStrategy(seed=0), top_k=20, transaction_cost_bps=5)
result_random = bt_random.run()

print(f"Random strategy IS Sharpe  : {result_random.sharpe:.3f}")
print(f"Random strategy IS Ann Ret : {result_random.annualized_return:.2%}")

In [ ]:
# Helper: plot equity curve
def plot_equity(result, title='Strategy equity curve'):
    daily_ret = np.array(result.daily_returns)
    equity = (1 + daily_ret).cumprod() - 1
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(equity * 100, lw=1.5)
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_title(title)
    ax.set_xlabel('Trading day')
    ax.set_ylabel('P&L (%)')
    plt.tight_layout()
    plt.show()

plot_equity(result_random, 'Random strategy equity (noise baseline)')

---
## Part 3: Find a Real Signal

The market has several features.  Exactly one is a planted alpha — it has a *small but real* predictive relationship with next-day returns.  The others are pure noise.

A good way to measure predictiveness is the **Information Coefficient (IC)**: the Spearman rank-correlation between your signal today and tomorrow's cross-sectional returns.  A mean IC > 0 means the signal is directionally useful.

In [ ]:
from scipy.stats import spearmanr

def rolling_ic(features_3d, returns_2d, feature_idx, lag=1):
    """Compute daily IC for a given feature index."""
    ics = []
    for t in range(min(len(features_3d), len(returns_2d)) - lag):
        signal = features_3d[t, :, feature_idx]
        fwd_ret = returns_2d[t + lag]
        corr, _ = spearmanr(signal, fwd_ret)
        ics.append(corr)
    return np.array(ics)


fig, axes = plt.subplots(len(market.FEATURE_NAMES), 1, figsize=(10, 2 * len(market.FEATURE_NAMES)))
summary = {}

for i, feat_name in enumerate(market.FEATURE_NAMES):
    ics = rolling_ic(train_features, train_returns, i)
    mean_ic = np.mean(ics)
    icir = mean_ic / (np.std(ics) + 1e-9) * np.sqrt(252)
    summary[feat_name] = {'mean_ic': mean_ic, 'IC-IR': icir}

    ax = axes[i] if len(market.FEATURE_NAMES) > 1 else axes
    ax.plot(pd.Series(ics).rolling(20).mean(), lw=1.2)
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_title(f'{feat_name}  |  mean IC={mean_ic:.4f}  |  IC-IR={icir:.2f}')
    ax.set_xlabel('Day')

plt.tight_layout()
plt.show()

print("\nFeature IC summary (IS)")
print(pd.DataFrame(summary).T.to_string(float_format='{:.4f}'.format))

**Exercise 3.1** — Which feature has the highest mean IC?  Is it statistically significant (|IC-IR| > 1.5)?

In [ ]:
# Fill in the feature name with the highest mean IC from the chart above
best_feature = market.FEATURE_NAMES[0]  # e.g. 'mom_1m' — replace with your answer
best_feature_idx = market.FEATURE_NAMES.index(best_feature)
print(f"Selected: {best_feature} (index {best_feature_idx})")

---
## Part 4: The Overfitting Trap

Now the temptation: what if you pick **not** the feature with the best *true* IC, but the one with the best IS Sharpe after exhaustively testing every combination?

We'll systematically try all feature subsets + three different long/short cutoffs.  This is also called **data snooping** or **p-hacking**.

In [ ]:
class WeightedStrategy:
    """Linear combination of features with learnable weights."""
    def __init__(self, weights: np.ndarray):
        self.weights = weights / (np.linalg.norm(weights) + 1e-8)

    def predict(self, features: np.ndarray) -> np.ndarray:
        return features @ self.weights


n_features = len(market.FEATURE_NAMES)
results_grid = []

print("Running grid search (this is the overfitting trap)...")
for top_k in [10, 20, 40]:
    for _ in range(200):   # 200 random weight vectors
        w = np.random.default_rng(len(results_grid)).standard_normal(n_features)
        strat = WeightedStrategy(w)
        bt = SimpleBacktest(market, strat, top_k=top_k, transaction_cost_bps=5)
        r = bt.run()
        results_grid.append({'weights': w, 'top_k': top_k, 'sharpe': r.sharpe, 'ann_ret': r.annualized_return})

df_grid = pd.DataFrame(results_grid)
best_row = df_grid.loc[df_grid['sharpe'].idxmax()]
print(f"\nBest IS Sharpe from grid: {best_row['sharpe']:.3f}  (top_k={best_row['top_k']:.0f})")
print(f"Number of strategies tested: {len(df_grid)}")

In [ ]:
# Distribution of IS Sharpes from the grid
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df_grid['sharpe'], bins=40, edgecolor='white', color='steelblue', alpha=0.8)
ax.axvline(best_row['sharpe'], color='red', lw=2, label=f"Best selected: {best_row['sharpe']:.2f}")
ax.set_title('Distribution of IS Sharpe ratios across the search grid')
ax.set_xlabel('IS Sharpe')
ax.legend()
plt.tight_layout()
plt.show()

**The trap:** the best IS Sharpe looks impressive — but it was *selected* from 600 trials.  By chance alone, some weights will look great in-sample.  Let's see what the grader thinks.

---
## Part 5: Submit to the Grader

The grader runs your strategy on the **hidden holdout** period (the final 250 days you have never seen).  It reports:

- **OOS Sharpe** — the Sharpe ratio on holdout returns
- **OOS IC-IR** — average IC / std(IC) × √252 on the holdout
- **Alpha discovery score** — contribution to the hidden planted alpha (0–10 scale)

You can submit from this notebook **or** from the web platform at `/compete`.

### 5A: Submit the grid-search "winner" (the overfit strategy)

In [ ]:
best_weights = best_row['weights']

# Build the submission string — paste this into the web editor or use the API
weights_repr = ', '.join(f'{w:.6f}' for w in best_weights)
submission_code = f'''\
import numpy as np

class MyStrategy:
    def __init__(self):
        self._weights = np.array([{weights_repr}])
        self._weights /= np.linalg.norm(self._weights) + 1e-8

    def predict(self, features: np.ndarray) -> np.ndarray:
        return features @ self._weights
'''


from convexpi.lab import submit
import os

COMPETITION = "demo-fall-2026"

# One-line submission. Set your key once (get it at
# https://www.convexpi.ai/settings/api-keys):
#     os.environ["CONVEXPI_API_KEY"] = "cpk_live_…"
if os.environ.get("CONVEXPI_API_KEY"):
    submit(submission_code, competition=COMPETITION, name="overfit-gridsearch")
else:
    print("No CONVEXPI_API_KEY set — two ways to submit:")
    print("  1) Get a key at https://www.convexpi.ai/settings/api-keys,")
    print("     set os.environ[\"CONVEXPI_API_KEY\"], and re-run this cell.")
    print(f"  2) Or paste the code below into the web editor:")
    print(f"     https://www.convexpi.ai/compete/{COMPETITION}/submit")
    print()
    print(submission_code)


After submitting, wait ~30 seconds and check your grade report on the leaderboard.  Note the OOS Sharpe.

**Exercise 5.1** — Is the OOS Sharpe higher or lower than the IS Sharpe?  Why?

---
### 5B: Submit the principled strategy (use the signal with highest IC)

Now let's build a strategy based on what we learned from the IC analysis.

In [ ]:
# Fill in the best feature index you found in Exercise 3.1
BEST_FEATURE_IDX = 0   # <-- change this

principled_code = f'''\
import numpy as np

class MyStrategy:
    """
    Single-factor strategy using the feature with the highest in-sample IC.
    No parameter tuning beyond the feature selection.
    """
    FEATURE_IDX = {BEST_FEATURE_IDX}

    def predict(self, features: np.ndarray) -> np.ndarray:
        raw = features[:, self.FEATURE_IDX]
        # Cross-sectional z-score for robustness
        mu, sigma = raw.mean(), raw.std() + 1e-8
        return (raw - mu) / sigma
'''


from convexpi.lab import submit
import os

COMPETITION = "demo-fall-2026"

# One-line submission. Set your key once (get it at
# https://www.convexpi.ai/settings/api-keys):
#     os.environ["CONVEXPI_API_KEY"] = "cpk_live_…"
if os.environ.get("CONVEXPI_API_KEY"):
    submit(principled_code, competition=COMPETITION, name="principled-single-factor")
else:
    print("No CONVEXPI_API_KEY set — two ways to submit:")
    print("  1) Get a key at https://www.convexpi.ai/settings/api-keys,")
    print("     set os.environ[\"CONVEXPI_API_KEY\"], and re-run this cell.")
    print(f"  2) Or paste the code below into the web editor:")
    print(f"     https://www.convexpi.ai/compete/{COMPETITION}/submit")
    print()
    print(principled_code)


Submit this second strategy and compare the grade reports.

**Exercise 5.2** — Which strategy had a higher OOS Sharpe?  What does this tell you about the relationship between IS Sharpe and OOS performance?

---
## Part 6: Interpret Your Grade Report

The grader returns a JSON report.  Let's walk through the key fields.

In [ ]:
# Example grade report structure (replace with your actual report from the API or web UI)
example_report = {
    "oos_sharpe": 0.42,
    "oos_annualized_return": 0.031,
    "oos_max_drawdown": -0.087,
    "alpha_discoveries": {
        "planted_alpha": 2.3,
        "noise_signals": 0.0,
    },
    "oos_contribution": 2.3,
    "noise_contributions": {"noise_0": 0.1, "noise_1": -0.05},
}

print("Field                   Value    Meaning")
print("-" * 60)
print(f"oos_sharpe              {example_report['oos_sharpe']:.3f}    Annualised Sharpe on holdout data")
print(f"oos_annualized_return   {example_report['oos_annualized_return']:.3f}    Compound annual return on holdout")
print(f"oos_max_drawdown        {example_report['oos_max_drawdown']:.3f}    Worst peak-to-trough decline")
print(f"oos_contribution        {example_report['oos_contribution']:.3f}    IC-IR attributable to planted alpha")
print("                                 (clamped to [-10, 10]; closer to 10 = better alpha recovery)")

**What to look for:**

| OOS Sharpe | Interpretation |
|---|---|
| < 0 | Strategy loses money OOS — likely overfit |
| 0 – 0.5 | Weak alpha or noisy; could be real or lucky |
| 0.5 – 1.5 | Decent for a real strategy with no look-ahead |
| > 1.5 | Suspiciously high — double-check for data leaks |

The **oos_contribution** measures how much of the planted alpha signal your strategy captured, regardless of overall Sharpe.  A strategy with contribution = 8 found most of the planted signal even if overall Sharpe was modest due to transaction costs.

---
## Part 7: Challenges

Work through as many as you can.  Discuss your findings with classmates — compare grade reports.

**Challenge A (Easy):** The z-score normalisation in Part 5B is one robustness technique.  Try replacing it with a rank transformation (use `scipy.stats.rankdata`).  Does OOS Sharpe improve?

**Challenge B (Medium):** Implement an exponential moving average (EMA) of the signal instead of using the raw daily value.  Experiment with different half-lives (5, 10, 20 days).  What happens to transaction costs and IC as you increase the half-life?

**Challenge C (Medium):** Combine two features using a weighted average.  Find the weights using the in-sample data (e.g., OLS on lagged features → next-day return).  Does the combination beat the best single feature OOS?

**Challenge D (Hard):** Implement a walk-forward optimisation: fit weights on a rolling 100-day window, predict on the next day, and repeat.  Does walk-forward outperform the static weights?  What does this tell you about regime stationarity?

In [ ]:
# Challenge A starter
from scipy.stats import rankdata

class RankedStrategy:
    FEATURE_IDX = BEST_FEATURE_IDX

    def predict(self, features: np.ndarray) -> np.ndarray:
        raw = features[:, self.FEATURE_IDX]
        ranks = rankdata(raw)  # 1 to n_assets
        # Center around zero
        return ranks - ranks.mean()

bt_ranked = SimpleBacktest(market, RankedStrategy(), top_k=20, transaction_cost_bps=5)
r_ranked = bt_ranked.run()
print(f"Ranked strategy IS Sharpe: {r_ranked.sharpe:.3f}")

In [ ]:
# Challenge B starter — EMA signal
class EMAStrategy:
    def __init__(self, feature_idx: int, half_life: int = 10):
        self.feature_idx = feature_idx
        self.alpha = 1 - np.exp(-np.log(2) / half_life)
        self._ema = None

    def predict(self, features: np.ndarray) -> np.ndarray:
        raw = features[:, self.feature_idx]
        if self._ema is None:
            self._ema = raw.copy()
        else:
            self._ema = self.alpha * raw + (1 - self.alpha) * self._ema
        mu, sigma = self._ema.mean(), self._ema.std() + 1e-8
        return (self._ema - mu) / sigma

for hl in [5, 10, 20]:
    strat = EMAStrategy(BEST_FEATURE_IDX, half_life=hl)
    bt = SimpleBacktest(market, strat, top_k=20, transaction_cost_bps=5)
    r = bt.run()
    print(f"EMA half_life={hl:2d}  IS Sharpe={r.sharpe:.3f}  Ann Ret={r.annualized_return:.2%}")

---
## Wrap-up

Key takeaways from this mission:

1. **IS Sharpe is not OOS Sharpe.** Searching over many strategies inflates IS performance. The more degrees of freedom you use, the larger the gap.

2. **IC is a better guide than Sharpe** when selecting signals, because it measures the predictive relationship directly, without the confound of portfolio construction choices.

3. **Simple and robust beats complex and tuned.** A single-factor z-score strategy built from first principles often outperforms an elaborate grid-search winner OOS.

4. **Walk-forward validation** is the closest we can get to true OOS testing during development — but it is no substitute for actually holding out data you have never touched.

---

*Mission 2: Market-Maker Arena → coming next week*